# FaceProof 01 — Preprocess (compression matching) + Day 1 gate

**What this notebook does (Day 1 of the plan):**
1. Clone the repo and install deps on Colab.
2. Download the datasets (140k real+fake faces, SFHQ Part 2 = Stable Diffusion v1.4).
3. Run preprocessing: route **every** image through identical crop + resize + JPEG re-encode
   so compression history cannot become a label proxy ("compression matching").
4. **Day 1 gate:** load a *matched* real/fake batch and view it.

The notebook only *calls* functions from `src/` — all the real logic lives there, so every line
is traceable to code you can read and defend. The DCT leakage check (cells at the bottom) is
**Day 2**, kept here because they share the same data setup.

> Runtime: **Runtime ▸ Change runtime type ▸ GPU** is optional today (Day 1 is CPU image work).

## 1. Setup — clone repo + install deps

In [ ]:
# >>> SET THIS to your GitHub repo URL (the one you pushed faceproof to) <<<
REPO_URL = "https://github.com/Ezed9/faceproof.git"

!git clone $REPO_URL
%cd faceproof

# Colab already has torch/torchvision/pandas/pillow/sklearn/matplotlib. Just top up the small ones.
# NOTE: we intentionally do NOT install facenet-pytorch today -> preprocessing falls back to a
# fast center-crop, which is fine because these datasets are already face-aligned.
!pip install -q pyyaml tqdm

import sys, os
sys.path.append(os.getcwd())   # so `from src.preprocessing import ...` works
print("cwd:", os.getcwd())

## 2. Persistence — mount Drive

Raw downloads go to `/content` (wiped when the session ends). The **crops** (our processed faces)
go to **Drive** so you only preprocess once and reuse them in later notebooks.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

CROPS_ROOT = Path('/content/drive/MyDrive/faceproof/data/crops')  # persistent cache
CROPS_ROOT.mkdir(parents=True, exist_ok=True)
RAW = Path('/content/data/raw'); RAW.mkdir(parents=True, exist_ok=True)
print('crops ->', CROPS_ROOT)
print('raw   ->', RAW)

## 3. Download datasets (Kaggle API)

Get your token: **kaggle.com ▸ Account ▸ Create New API Token** → downloads `kaggle.json`.
Run the cell, then upload that file when prompted.

Datasets (verify the slugs on Kaggle if a download 404s — Kaggle slugs occasionally change):
- `xhlulu/140k-real-and-fake-faces` — real (FFHQ) + fake (StyleGAN)
- `selfishgene/synthetic-faces-high-quality-sfhq-part-2` — SFHQ Part 2 (SD v1.4)

In [ ]:
!pip install -q kaggle
from google.colab import files
print('Upload your kaggle.json:')
files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
print('kaggle credentials installed.')

In [ ]:
# ~4-5 GB total; takes a few minutes. --unzip extracts in place.
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p /content/data/raw --unzip
!kaggle datasets download -d selfishgene/synthetic-faces-high-quality-sfhq-part-2 -p /content/data/raw --unzip
print('done.')

## 4. Locate the image folders

Folder layouts can differ from what we expect, so we **print the biggest image folders** and you
confirm the three source paths below. Set `REAL_DIR`, `STYLEGAN_DIR`, `SD_DIR` from the printout.

In [ ]:
from collections import Counter
EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
counts = Counter()
for p in RAW.rglob('*'):
    if p.suffix.lower() in EXT:
        counts[p.parent] += 1
print('Largest image folders (count, path):')
for d, c in sorted(counts.items(), key=lambda x: -x[1])[:20]:
    print(f'{c:>7}  {d}')

In [ ]:
# Confirm these against the printout above and edit if needed.
# The 140k set nests train/valid/test; the train split alone is plenty for now.
REAL_DIR     = RAW / 'real_vs_fake/real-vs-fake/train/real'   # FFHQ reals  (label 0)
STYLEGAN_DIR = RAW / 'real_vs_fake/real-vs-fake/train/fake'   # StyleGAN    (label 1)

# SFHQ Part 2 = the big non-140k folder in the printout (Stable Diffusion v1.4, label 1).
# Auto-guess: largest image folder whose path does NOT contain 'real_vs_fake'.
SD_DIR = next(d for d, _ in sorted(counts.items(), key=lambda x: -x[1])
              if 'real_vs_fake' not in str(d))

for name, d in [('REAL_DIR', REAL_DIR), ('STYLEGAN_DIR', STYLEGAN_DIR), ('SD_DIR', SD_DIR)]:
    n = len([p for p in Path(d).rglob('*') if p.suffix.lower() in EXT])
    print(f'{name:12} {d}  ({n} images)')

## 5. Preprocess = compression matching

`preprocess_folder` (in `src/preprocessing.py`) does, per image: load → square crop → resize to
`image_size` → **re-encode as JPEG at one fixed quality**. Because real and fake go through the
*identical* pipeline, the detector can't cheat on resolution / JPEG artifacts — it has to learn
actual synthesis cues. (`use_mtcnn=False` → center-crop, fast; flip to True later if you want
MTCNN alignment and have installed `facenet-pytorch`.)

`image_size` and `jpeg_match_quality` are read from `configs/data.yaml` — no hard-coded settings.

In [ ]:
import yaml
from src.preprocessing import preprocess_folder

cfg = yaml.safe_load(open('configs/data.yaml'))
IMG = cfg['image_size']            # 224
Q   = cfg['jpeg_match_quality']    # 90
N_PER_CLASS = 6000                 # Day 1 subset. Raise (or set None for all) when caching features.
print(f'image_size={IMG}  jpeg_quality={Q}  n_per_class={N_PER_CLASS}')

preprocess_folder(REAL_DIR,     CROPS_ROOT / 'real',     IMG, Q, use_mtcnn=False, limit=N_PER_CLASS)
preprocess_folder(STYLEGAN_DIR, CROPS_ROOT / 'stylegan', IMG, Q, use_mtcnn=False, limit=N_PER_CLASS)
preprocess_folder(SD_DIR,       CROPS_ROOT / 'sd',       IMG, Q, use_mtcnn=False, limit=N_PER_CLASS)

## 6. Build manifest + leakage-safe splits

Build the manifest **from the crops** (everything downstream uses matched crops). `make_splits`
forces the unseen generator (SD) into the **test** split only.

> Honest note: `validate_splits` in `src/data.py` flags any *source* that appears in more than one
> split. `real` legitimately appears in train/val/test (we want reals everywhere), so it will
> warn — that's expected, not a bug in our data. The check that actually matters for leakage is
> the assert below: **the unseen generator must never appear outside `test`.**

In [ ]:
from src.data import build_manifest, make_splits

sources = {
    'real':     {'dir': str(CROPS_ROOT / 'real'),     'label': 0, 'generator': 'real'},
    'stylegan': {'dir': str(CROPS_ROOT / 'stylegan'), 'label': 1, 'generator': 'stylegan'},
    'sd':       {'dir': str(CROPS_ROOT / 'sd'),       'label': 1, 'generator': 'stable_diffusion'},
}
df = build_manifest(sources, out_csv='data/manifest.csv')
df = make_splits(df, train_generators=['stylegan'], test_unseen=['stable_diffusion'],
                 val_fraction=0.15, test_fraction=0.15, seed=13)

unseen_splits = set(df.loc[df.generator == 'stable_diffusion', 'split'])
assert unseen_splits <= {'test'}, f'LEAKAGE: SD appears in {unseen_splits}, must be test-only'
print('OK: unseen generator (SD) is test-only ->', unseen_splits)

## 7. ✅ Day 1 gate — a matched real/fake batch loads and renders

Success = a tensor of shape `(2k, 3, 224, 224)` in `[0, 1]` plus a grid showing reals (top) and
fakes (bottom), all the same size and visibly face crops.

In [ ]:
import torch, matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as T

to_tensor = T.Compose([T.Resize((IMG, IMG)), T.ToTensor()])  # -> (3,H,W) float in [0,1]

def sample_batch(df, label, k):
    paths = df[df.label == label].sample(k, random_state=0).path.tolist()
    imgs = torch.stack([to_tensor(Image.open(p).convert('RGB')) for p in paths])
    return imgs, paths

k = 8
real_b, _ = sample_batch(df, 0, k)
fake_b, _ = sample_batch(df, 1, k)
batch = torch.cat([real_b, fake_b])

print('batch:', tuple(batch.shape), batch.dtype, 'range [%.2f, %.2f]' % (batch.min(), batch.max()))
assert batch.shape[1:] == (3, IMG, IMG) and 0.0 <= batch.min() and batch.max() <= 1.0
print('GATE PASSED ✅  matched real/fake batch loads')

fig, axes = plt.subplots(2, k, figsize=(2 * k, 4.2))
for i in range(k):
    axes[0, i].imshow(real_b[i].permute(1, 2, 0)); axes[0, i].axis('off')
    axes[1, i].imshow(fake_b[i].permute(1, 2, 0)); axes[1, i].axis('off')
axes[0, 0].set_ylabel('real', fontsize=12); axes[1, 0].set_ylabel('fake', fontsize=12)
fig.suptitle('Day 1 gate: matched crops (top=real, bottom=fake)'); plt.tight_layout(); plt.show()

---
**Log it.** Add to `experiments.md` under Day 1: what ran, the crop size/quality, `N_PER_CLASS`,
and that the gate passed. Commit `data/manifest.csv` is gitignored — that's fine; crops live on Drive.

## Day 2 (next) — DCT leakage check
Run `src/leakage_check.py` on ~500/class from the crops. If DCT+SVM separates real/fake with
accuracy **> ~0.80**, preprocessing is leaking (compression history still differs) and must be
fixed before trusting any detector result. **Run the leakage-check cell below.**

## 8. ✅ Day 2 gate (part 1) — DCT leakage check

`run_leakage_check` (in `src/leakage_check.py`) trains a *deliberately dumb* classifier — a 2D-DCT
log-spectrum fed to a linear SVM — on ~500 real vs 500 fake crops. The DCT spectrum captures
compression / resampling fingerprints, **not** face semantics. So:

- **Accuracy near chance (~0.50)** → good. The compression matching worked; no obvious shortcut.
- **Accuracy high (> ~0.80)** → BAD. Real and fake still differ in some artifact (resolution, JPEG
  history). Any later 'detector' would be cheating — fix preprocessing before trusting results.

We check the **training pair** (real vs StyleGAN) because that's what the detector learns from.

In [ ]:
from src.leakage_check import run_leakage_check

# ~500 per class from the matched crops. Near chance (~0.50) = good.
acc = run_leakage_check(CROPS_ROOT / 'real', CROPS_ROOT / 'stylegan', n_per_class=500)

print()
if acc > 0.80:
    print(f'⚠️  LEAKAGE: DCT+SVM = {acc:.3f}. Preprocessing is leaking a shortcut.')
    print('    Do NOT trust any detector result until this is near chance.')
else:
    print(f'✅ Day 2 leakage gate PASSED: DCT+SVM = {acc:.3f} (near chance).')
    print('    Compression matching is working — safe to extract features (notebook 02).')

# Optional: also sanity-check the unseen test pair (real vs SD). Same interpretation.
# acc_sd = run_leakage_check(CROPS_ROOT / 'real', CROPS_ROOT / 'sd', n_per_class=500)